# VectorMIDE Colab Runner

这个 notebook 用来在 Colab 上运行 `train/train_vector_offline.py`。Colab 默认目录是 `/content`，所以必须先把项目放到 Colab 能访问的位置，然后 `cd` 到项目根目录。

In [4]:
# 1. 查看 Colab GPU
!nvidia-smi

import os, sys, pathlib
print('cwd =', os.getcwd())
print('python =', sys.executable)

/bin/bash: line 1: nvidia-smi: command not found
cwd = /content
python = /usr/bin/python3


## 选择项目位置

如果你把整个 `DeepMide_v2` 上传到了 Google Drive，运行下面这个 cell 并把 `PROJECT_DIR` 改成你的实际路径。这个目录下面应该能看到 `train/`, `model/`, `yml_files/`, `data/`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/DeepMide_v2"

project = pathlib.Path(PROJECT_DIR)
print("project exists:", project.exists())
print("train file exists:", (project / "train" / "train_vector_offline.py").exists())
print("config exists:", (project / "yml_files" / "VectorMIDE_cuda.yaml").exists())
print("data exists:", (project / "data").exists())

%cd "{PROJECT_DIR}"
!pwd
!ls -la


MessageError: [dfs_ephemeral] Credentials propagation unsuccessful

## 安装依赖

Colab 通常已经有 PyTorch CUDA 版本；这里主要安装项目依赖。

In [ ]:
# 3. 安装依赖
!python -m pip install -q -r requirements.txt

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 快速检查

先用 `--dry-run` 确认数据、配置、模型都能连起来。

In [ ]:
# 4. Dry run
!python train/train_vector_offline.py \
  --config yml_files/VectorMIDE_cuda.yaml \
  --limit 128 \
  --dry-run \
  --device cuda

## 正式训练

In [ ]:
# 5. 正式训练
!python train/train_vector_offline.py \
  --config yml_files/VectorMIDE_cuda.yaml \
  --device cuda

## 12-step evaluation

In [ ]:
# 6. 多步评估
!python train/evaluate_vector.py \
  --config yml_files/VectorMIDE_cuda.yaml \
  --checkpoint outputs/cnn_transformer_new/vector_mide_offline_cuda.pt \
  --split online \
  --forecast-horizon 12 \
  --output-dir outputs/cnn_transformer_new/eval_online_h12 \
  --device cuda